# Лабораторная работа 2

In [ ]:
import pandas as pd
df = pd.read_csv('person_2025_update.csv.bz2')
df

,id,wd_id,wp_id,slug,name,occupation,prob_ratio,gender,twitter,alive,...,deathdate,deathyear,bplace_geacron_name,dplace_geacron_name,is_group,l_,age,non_en_page_views,coefficient_of_variation,hpi
0,18934,Q9458,18934,Muhammad,Muhammad,RELIGIOUS FIGURE,0.000000,M,NaN,False,...,0632-06-08,632.0,mecca,medina,False,26.649583,62.0,3732407,3.725079,100.000000
1,3395,Q9441,3395,Gautama_Buddha,Gautama Buddha,PHILOSOPHER,0.000000,M,NaN,False,...,NaN,-452.0,lumbini,kushinagar,False,31.940636,114.0,2038657,3.088739,99.481783
2,14627,Q935,14627,Isaac_Newton,Isaac Newton,PHYSICIST,0.000000,M,NaN,False,...,1727-03-31,1726.0,woolsthorpe-by-colsterworth,kensington,False,30.988775,83.0,2508822,3.782160,99.439201
3,4848272,Q22686,4848272,Donald_Trump,Donald Trump,POLITICIAN,0.000000,M,realDonaldTrump,True,...,NaN,NaN,queens,NaN,False,21.201757,79.0,17028468,4.193298,99.267069
4,17414699,Q720,17414699,Genghis_Khan,Genghis Khan,MILITARY PERSONNEL,0.000000,M,NaN,False,...,1227-08-18,1227.0,khentii-mountains,yinchuan,False,28.157479,65.0,2506097,3.040899,98.434181
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126577,68034901,Q107337715,68034901,Andrés_Madera,Andrés Madera,ATHLETE,5.661844,M,NaN,True,...,NaN,NaN,montreal,NaN,False,1.000000,37.0,1,1.000000,0.000000
126578,68220355,Q107616230,68220355,Maxime_Collard,Maxime Collard,ATHLETE,2.062645,F,NaN,True,...,NaN,NaN,lyon,NaN,False,1.000000,NaN,1,1.000000,0.000000
126579,68330746,Q107654642,68330746,Jaime_Mateu,Jaime Mateu,ATHLETE,5.214603,M,NaN,True,...,NaN,NaN,palma-de-mallorca,NaN,False,1.000000,30.0,1,1.000000,0.000000
126580,61744365,Q67264794,61744365,Wesley_Kitts,Wesley Kitts,ATHLETE,1.051450,M,NaN,True,...,NaN,NaN,knoxville-tennessee,NaN,False,1.000000,35.0,1,1.000000,0.000000


## Подготовим датасет

In [ ]:
cols_keep = ['id','age','non_en_page_views','hpi','gender']
cols_keep = [c for c in cols_keep if c in df.columns]

unique_df = df[df['age'].notna() & df['non_en_page_views'].notna()]
tidy = unique_df.groupby(['id','age'], as_index=False).first()[cols_keep]

tidy['id'] = tidy['id'].astype(int)
tidy['age'] = tidy['age'].astype(float)
tidy['non_en_page_views'] = tidy['non_en_page_views'].astype(int)
if 'alive' in tidy.columns:
    tidy['alive'] = tidy['alive'].astype(bool)
for col in ['hpi']:
    if col in tidy.columns:
        tidy[col] = pd.to_numeric(tidy[col], errors='coerce')

tidy = tidy.sort_values(['id','age']).reset_index(drop=True)
tidy['gender'] = tidy['gender'].fillna('unknown')

tidy.head()

,id,age,non_en_page_views,hpi,gender
0,307,56.0,2118601,87.871169,M
1,308,63.0,2064018,96.252463,M
2,339,77.0,455508,83.742192,F
3,340,78.0,18945,64.084124,M
4,344,96.0,6591,58.491034,M


## Функция для сравнения

In [ ]:
def check_equal_full(a, b):
    cols = a.columns.tolist()
    a2 = a.sort_values(['id']).reset_index(drop=True)[cols]
    b2 = b.sort_values(['id']).reset_index(drop=True)[cols]
    for c in cols:
        if c in a2.columns and c in b2.columns:
            try:
                b2[c] = b2[c].astype(a2[c].dtype)
            except Exception:
                pass
    return a2.equals(b2)

## Ломаем №1 - значения в именах стобцов

In [ ]:
tmp = tidy[['id','gender','non_en_page_views']].copy()  #срезаем ненужные столбцы

broken = tmp.pivot(index='id', columns='gender', values='non_en_page_views')
broken = broken[['F','M','unknown']] if set(['F','M','unknown']).issubset(broken.columns) else broken
broken.head()

gender,F,M,unknown
id,,,
307,NaN,2118601.0,NaN
308,NaN,2064018.0,NaN
339,455508.0,NaN,NaN
340,NaN,18945.0,NaN
344,NaN,6591.0,NaN


## Восстанавливаем

In [ ]:
restored = (broken.reset_index()
                  .melt(id_vars='id', var_name='gender', value_name='non_en_page_views')
                  .dropna()
                  .sort_values(['id','gender'])
                  .reset_index(drop=True))

print("Совпадают ли таблицы:", check_equal_full(tmp, restored))
tmp


Совпадают ли таблицы: True


,id,gender,non_en_page_views
0,307,M,2118601
1,308,M,2064018
2,339,F,455508
3,340,M,18945
4,344,M,6591
...,...,...,...
125024,78378148,F,91197
125025,78639405,M,209133
125026,79161145,M,1
125027,79403002,M,1


## Ломаем №2 - несколько значений в строке

In [ ]:
packed = tidy[['id']].copy()
packed['bundle'] = (
    tidy['age'].astype(str).replace('nan', '') + '|' +
    tidy['non_en_page_views'].astype(str) + '|' +
    tidy['hpi'].astype(str) + '|' +
    tidy['gender'].fillna('')
)
packed

,id,bundle
0,307,56.0|2118601|87.871169|M
1,308,63.0|2064018|96.252463|M
2,339,77.0|455508|83.742192|F
3,340,78.0|18945|64.084124|M
4,344,96.0|6591|58.491034|M
...,...,...
125024,78378148,22.0|91197|43.52610084733002|F
125025,78639405,22.0|209133|44.7224042725329|M
125026,79161145,19.0|1|2.625304421043568|M
125027,79403002,16.0|1|2.023206582855801|M


## Восстанавливаем

In [ ]:
unpacked = packed['bundle'].str.split('|', expand=True)
unpacked.columns = ['age','non_en_page_views','hpi','gender']

restored = pd.concat([packed[['id']], unpacked], axis=1)
restored = restored.replace({'': np.nan}).astype({
    'id': 'int64',
    'age': 'float64',
    'non_en_page_views': 'int64',
    'hpi': 'float64',
    'gender': 'object'
}).sort_values('id').reset_index(drop=True)

print("Совпадают ли таблицы:", check_equal_full(tidy, restored))
restored.head()


Совпадают ли таблицы: True


,id,age,non_en_page_views,hpi,gender
0,307,56.0,2118601,87.871169,M
1,308,63.0,2064018,96.252463,M
2,339,77.0,455508,83.742192,F
3,340,78.0,18945,64.084124,M
4,344,96.0,6591,58.491034,M


## Ломаем №3 - переменные находятся и в именах столбцов и в значениях

In [ ]:
messy3 = tidy.melt(id_vars=['id','gender'], value_vars=['age','non_en_page_views','hpi'], var_name='metric', value_name='val')
messy3 = messy3.pivot(index=['id','metric'], columns='gender', values='val')
messy3


gender                       F             M  unknown
id       metric                                      
307      age               NaN  5.600000e+01      NaN
         hpi               NaN  8.787117e+01      NaN
         non_en_page_views NaN  2.118601e+06      NaN
308      age               NaN  6.300000e+01      NaN
         hpi               NaN  9.625246e+01      NaN
...                         ..           ...      ...
79403002 hpi               NaN  2.023207e+00      NaN
         non_en_page_views NaN  1.000000e+00      NaN
80456484 age               NaN  1.800000e+01      NaN
         hpi               NaN  2.802670e-01      NaN
         non_en_page_views NaN  1.000000e+00      NaN

[375087 rows x 3 columns]

## Восстанавливаем

In [ ]:
restored = messy3.stack().reset_index(name='val').pivot(index=['id','gender'], columns='metric', values='val').reset_index()
restored = restored[cols_keep].sort_values(['id','gender']).reset_index(drop=True)
print("Совпадают ли таблицы:", check_equal_full(tidy, restored))
restored.head()


Совпадают ли таблицы: True


metric,id,age,non_en_page_views,hpi,gender
0,307,56.0,2118601.0,87.871169,M
1,308,63.0,2064018.0,96.252463,M
2,339,77.0,455508.0,83.742192,F
3,340,78.0,18945.0,64.084124,M
4,344,96.0,6591.0,58.491034,M


## Сводная таблица

In [ ]:
aggr_df = tidy.groupby(['age', 'gender']).agg({
    'age': 'mean',
    'non_en_page_views': 'sum'
})
tabel = aggr_df.unstack('gender')

tabel.columns = tabel.columns.map('_'.join)

tabel = tabel.reset_index()

tabel.head()

,age,age_F,age_M,age_unknown,non_en_page_views_F,non_en_page_views_M,non_en_page_views_unknown
0,-1700.0,NaN,-1700.0,NaN,NaN,1433.0,NaN
1,-875.0,NaN,-875.0,NaN,NaN,16939.0,NaN
2,-750.0,NaN,-750.0,NaN,NaN,16314.0,NaN
3,-599.0,-599.0,NaN,NaN,1618.0,NaN,NaN
4,-502.0,NaN,-502.0,NaN,NaN,63159.0,NaN
